In [ ]:
"""
============================================================
 LIBRARY MANAGEMENT SYSTEM
============================================================
A menu-driven console application to manage a library's book
catalog: add, search, update, delete, issue, and return books.

Concepts demonstrated:
- Object-Oriented Programming (Book, Library classes)
- Lists & Dictionaries (in-memory catalog storage)
- Loops & Conditional statements (menu handling, searching)
- Functions (modular design for every operation)
- File Handling (JSON persistence so data survives restarts)
============================================================
"""

import json
import os
from datetime import date

DATA_FILE = "library_data.json"


# ------------------------------------------------------------
# BOOK CLASS
# ------------------------------------------------------------
class Book:
    """Represents a single book record in the library."""11
    

    def __init__(self, book_id, title, author, genre, total_copies,
                 available_copies=None, issued_to=None):
        self.book_id = book_id
        self.title = title
        self.author = author
        self.genre = genre
        self.total_copies = int(total_copies)
        # If not provided (new book), all copies start as available
        self.available_copies = (
            int(available_copies) if available_copies is not None else int(total_copies)
        )
        # issued_to: dict mapping student_name -> issue_date (supports multiple copies issued)
        self.issued_to = issued_to if issued_to is not None else {}

    def to_dict(self):
        """Convert Book object to a dictionary (for JSON storage)."""
        return {
            "book_id": self.book_id,
            "title": self.title,
            "author": self.author,
            "genre": self.genre,
            "total_copies": self.total_copies,
            "available_copies": self.available_copies,
            "issued_to": self.issued_to,
        }

    @staticmethod
    def from_dict(data):
        """Create a Book object from a dictionary."""
        return Book(
            data["book_id"],
            data["title"],
            data["author"],
            data["genre"],
            data["total_copies"],
            data["available_copies"],
            data["issued_to"],
        )

    def __str__(self):
        status = "Available" if self.available_copies > 0 else "Not Available"
        return (
            f"{self.book_id:<8}{self.title[:25]:<27}{self.author[:18]:<20}"
            f"{self.genre[:12]:<14}{self.available_copies}/{self.total_copies:<8}{status}"
        )


# ------------------------------------------------------------
# LIBRARY CLASS
# ------------------------------------------------------------
class Library:
    """Manages the entire book catalog and all library operations."""

    def __init__(self, data_file=DATA_FILE):
        self.data_file = data_file
        self.books = {}          # book_id -> Book object
        self.next_id = 1
        self.load_data()

    # ---------------- File Handling ----------------
    def load_data(self):
        """Load book records from a JSON file, if it exists."""
        if os.path.exists(self.data_file):
            try:
                with open(self.data_file, "r") as f:
                    raw = json.load(f)
                for item in raw:
                    book = Book.from_dict(item)
                    self.books[book.book_id] = book
                if self.books:
                    max_id = max(int(bid[1:]) for bid in self.books if bid.startswith("B"))
                    self.next_id = max_id + 1
            except (json.JSONDecodeError, KeyError, ValueError):
                print("Warning: Could not read existing data file. Starting fresh.")
        else:
            # Seed with a couple of sample books so the menu isn't empty on first run
            self._seed_sample_data()

    def save_data(self):
        """Persist all book records to a JSON file."""
        with open(self.data_file, "w") as f:
            json.dump([b.to_dict() for b in self.books.values()], f, indent=4)

    def _seed_sample_data(self):
        sample = [
            ("Python Programming Basics", "Guido van Rossum", "Technology", 3),
            ("Data Structures & Algorithms", "R. Sedgewick", "Technology", 2),
            ("The Silent Patient", "Alex Michaelides", "Fiction", 4),
        ]
        for title, author, genre, copies in sample:
            self.add_book(title, author, genre, copies, silent=True)

    # ---------------- Core Operations ----------------
    def generate_id(self):
        new_id = f"B{self.next_id:04d}"
        self.next_id += 1
        return new_id

    def add_book(self, title, author, genre, total_copies, silent=False):
        """1. Add a new book to the library."""
        book_id = self.generate_id()
        book = Book(book_id, title, author, genre, total_copies)
        self.books[book_id] = book
        self.save_data()
        if not silent:
            print(f"\n✅ Book added successfully! Assigned Book ID: {book_id}")
        return book_id

    def display_all(self, book_list=None):
        """2. Display all books (or a filtered subset if provided)."""
        books = book_list if book_list is not None else list(self.books.values())
        if not books:
            print("\nNo books found.")
            return
        print("\n" + "-" * 95)
        print(f"{'ID':<8}{'Title':<27}{'Author':<20}{'Genre':<14}{'Copies':<10}{'Status'}")
        print("-" * 95)
        for book in books:
            print(book)
        print("-" * 95)
        print(f"Total records: {len(books)}")

    # ---------------- Search ----------------
    def search_by_id(self, book_id):
        book = self.books.get(book_id.strip().upper())
        return [book] if book else []

    def search_by_title(self, keyword):
        keyword = keyword.strip().lower()
        return [b for b in self.books.values() if keyword in b.title.lower()]

    def search_by_author(self, keyword):
        keyword = keyword.strip().lower()
        return [b for b in self.books.values() if keyword in b.author.lower()]

    # ---------------- Update ----------------
    def update_book(self, book_id, field, new_value):
        book = self.books.get(book_id)
        if not book:
            return False
        if field == "title":
            book.title = new_value
        elif field == "author":
            book.author = new_value
        elif field == "genre":
            book.genre = new_value
        elif field == "total_copies":
            new_total = int(new_value)
            diff = new_total - book.total_copies
            book.total_copies = new_total
            book.available_copies = max(0, book.available_copies + diff)
        else:
            return False
        self.save_data()
        return True

    # ---------------- Delete ----------------
    def delete_book(self, book_id):
        book = self.books.get(book_id)
        if not book:
            return False, "Book not found."
        if book.issued_to:
            return False, "Cannot delete: some copies are currently issued to students."
        del self.books[book_id]
        self.save_data()
        return True, "Book deleted successfully."

    # ---------------- Issue / Return ----------------
    def issue_book(self, book_id, student_name):
        book = self.books.get(book_id)
        if not book:
            return False, "Book not found."
        if book.available_copies <= 0:
            return False, "No copies available for issue right now."
        book.available_copies -= 1
        book.issued_to[student_name] = str(date.today())
        self.save_data()
        return True, f"'{book.title}' issued to {student_name} on {date.today()}."

    def return_book(self, book_id, student_name):
        book = self.books.get(book_id)
        if not book:
            return False, "Book not found."
        if student_name not in book.issued_to:
            return False, "No record of this book being issued to that student."
        del book.issued_to[student_name]
        book.available_copies = min(book.total_copies, book.available_copies + 1)
        self.save_data()
        return True, f"'{book.title}' returned successfully by {student_name}."

    # ---------------- Filtered Views ----------------
    def get_available_books(self):
        return [b for b in self.books.values() if b.available_copies > 0]

    def get_issued_books(self):
        return [b for b in self.books.values() if b.issued_to]

    def display_issued_details(self):
        issued = self.get_issued_books()
        if not issued:
            print("\nNo books are currently issued.")
            return
        print("\n" + "-" * 80)
        print(f"{'Book ID':<10}{'Title':<28}{'Student':<20}{'Issue Date'}")
        print("-" * 80)
        for book in issued:
            for student, issue_date in book.issued_to.items():
                print(f"{book.book_id:<10}{book.title[:26]:<28}{student[:18]:<20}{issue_date}")
        print("-" * 80)


# ------------------------------------------------------------
# INPUT HELPERS
# ------------------------------------------------------------
def get_nonempty_input(prompt):
    while True:
        value = input(prompt).strip()
        if value:
            return value
        print("This field cannot be empty. Please try again.")


def get_positive_int(prompt):
    while True:
        value = input(prompt).strip()
        if value.isdigit() and int(value) > 0:
            return int(value)
        print("Please enter a valid positive number.")


# ------------------------------------------------------------
# MENU-DRIVEN OPERATIONS
# ------------------------------------------------------------
def menu_add_book(library):
    print("\n--- Add New Book ---")
    title = get_nonempty_input("Enter Book Title: ")
    author = get_nonempty_input("Enter Author Name: ")
    genre = get_nonempty_input("Enter Genre/Category: ")
    copies = get_positive_int("Enter Number of Copies: ")
    library.add_book(title, author, genre, copies)


def menu_display_all(library):
    print("\n--- Full Catalog ---")
    library.display_all()


def menu_search(library):
    print("\n--- Search Book ---")
    print("1. By Book ID")
    print("2. By Title")
    print("3. By Author")
    choice = input("Choose search type (1-3): ").strip()

    if choice == "1":
        book_id = get_nonempty_input("Enter Book ID: ")
        results = library.search_by_id(book_id)
    elif choice == "2":
        keyword = get_nonempty_input("Enter Title (or part of it): ")
        results = library.search_by_title(keyword)
    elif choice == "3":
        keyword = get_nonempty_input("Enter Author Name (or part of it): ")
        results = library.search_by_author(keyword)
    else:
        print("Invalid choice.")
        return

    if results:
        library.display_all(results)
    else:
        print("\nNo matching books found.")


def menu_update_book(library):
    print("\n--- Update Book Details ---")
    book_id = get_nonempty_input("Enter Book ID to update: ").strip().upper()
    if book_id not in library.books:
        print("Book not found.")
        return

    print(library.books[book_id])
    print("\nWhich field do you want to update?")
    print("1. Title\n2. Author\n3. Genre\n4. Total Copies")
    choice = input("Choose (1-4): ").strip()
    field_map = {"1": "title", "2": "author", "3": "genre", "4": "total_copies"}
    field = field_map.get(choice)

    if not field:
        print("Invalid choice.")
        return

    if field == "total_copies":
        new_value = get_positive_int("Enter new total copies: ")
    else:
        new_value = get_nonempty_input(f"Enter new {field}: ")

    if library.update_book(book_id, field, new_value):
        print("✅ Book updated successfully.")
    else:
        print("Update failed.")


def menu_delete_book(library):
    print("\n--- Delete Book ---")
    book_id = get_nonempty_input("Enter Book ID to delete: ").strip().upper()
    confirm = input(f"Are you sure you want to delete {book_id}? (y/n): ").strip().lower()
    if confirm != "y":
        print("Deletion cancelled.")
        return
    success, message = library.delete_book(book_id)
    print(("✅ " if success else "❌ ") + message)


def menu_issue_book(library):
    print("\n--- Issue Book ---")
    book_id = get_nonempty_input("Enter Book ID: ").strip().upper()
    student_name = get_nonempty_input("Enter Student Name: ")
    success, message = library.issue_book(book_id, student_name)
    print(("✅ " if success else "❌ ") + message)


def menu_return_book(library):
    print("\n--- Return Book ---")
    book_id = get_nonempty_input("Enter Book ID: ").strip().upper()
    student_name = get_nonempty_input("Enter Student Name: ")
    success, message = library.return_book(book_id, student_name)
    print(("✅ " if success else "❌ ") + message)


def menu_display_available(library):
    print("\n--- Available Books ---")
    library.display_all(library.get_available_books())


def menu_display_issued(library):
    print("\n--- Issued Books ---")
    library.display_issued_details()


# ------------------------------------------------------------
# MAIN APPLICATION LOOP
# ------------------------------------------------------------
MENU_TEXT = """
============================================================
           LIBRARY MANAGEMENT SYSTEM
============================================================
 1. Add a New Book
 2. Display All Books
 3. Search for a Book (ID / Title / Author)
 4. Update Book Details
 5. Delete a Book
 6. Issue a Book to a Student
 7. Return an Issued Book
 8. Display Only Available Books
 9. Display Issued Books
10. Exit
============================================================
"""

MENU_ACTIONS = {
    "1": menu_add_book,
    "2": menu_display_all,
    "3": menu_search,
    "4": menu_update_book,
    "5": menu_delete_book,
    "6": menu_issue_book,
    "7": menu_return_book,
    "8": menu_display_available,
    "9": menu_display_issued,
}


def main():
    library = Library()
    print("Welcome to the Library Management System!")

    while True:
        print(MENU_TEXT)
        choice = input("Enter your choice (1-10): ").strip()

        if choice == "10":
            print("\nSaving data and exiting. Goodbye!")
            library.save_data()
            break

        action = MENU_ACTIONS.get(choice)
        if action:
            try:
                action(library)
            except Exception as e:
                print(f"An unexpected error occurred: {e}")
        else:
            print("Invalid choice. Please enter a number between 1 and 10.")


if __name__ == "__main__":
    main()

Welcome to the Library Management System!

           LIBRARY MANAGEMENT SYSTEM
 1. Add a New Book
 2. Display All Books
 3. Search for a Book (ID / Title / Author)
 4. Update Book Details
 5. Delete a Book
 6. Issue a Book to a Student
 7. Return an Issued Book
 8. Display Only Available Books
 9. Display Issued Books
10. Exit



Enter your choice (1-10):  2



--- Full Catalog ---

-----------------------------------------------------------------------------------------------
ID      Title                      Author              Genre         Copies    Status
-----------------------------------------------------------------------------------------------
B0001   Python Programming Basics  Guido van Rossum    Technology    3/3       Available
B0002   Data Structures & Algorit  R. Sedgewick        Technology    2/2       Available
B0003   The Silent Patient         Alex Michaelides    Fiction       4/4       Available
-----------------------------------------------------------------------------------------------
Total records: 3

           LIBRARY MANAGEMENT SYSTEM
 1. Add a New Book
 2. Display All Books
 3. Search for a Book (ID / Title / Author)
 4. Update Book Details
 5. Delete a Book
 6. Issue a Book to a Student
 7. Return an Issued Book
 8. Display Only Available Books
 9. Display Issued Books
10. Exit



Enter your choice (1-10):  6



--- Issue Book ---


...


--- Return Book ---


Enter Book ID:  B0002
Enter Student Name:  Shaun


✅ 'Data Structures & Algorithms' returned successfully by Shaun.

           LIBRARY MANAGEMENT SYSTEM
 1. Add a New Book
 2. Display All Books
 3. Search for a Book (ID / Title / Author)
 4. Update Book Details
 5. Delete a Book
 6. Issue a Book to a Student
 7. Return an Issued Book
 8. Display Only Available Books
 9. Display Issued Books
10. Exit



Enter your choice (1-10):  11


Invalid choice. Please enter a number between 1 and 10.

           LIBRARY MANAGEMENT SYSTEM
 1. Add a New Book
 2. Display All Books
 3. Search for a Book (ID / Title / Author)
 4. Update Book Details
 5. Delete a Book
 6. Issue a Book to a Student
 7. Return an Issued Book
 8. Display Only Available Books
 9. Display Issued Books
10. Exit

